# Etapa 2 — CNN construída do zero

Este notebook documenta a CNN manual usada para classificar imagens de gatos e cães. A rede é declarada camada a camada com PyTorch e não usa arquitetura pronta nem pesos pré-treinados. O notebook consome os mesmos módulos de produção do projeto para evitar divergência entre a explicação e o código executado.

**Contrato da etapa**

- entrada RGB `3×224×224`;
- seis convoluções, três operações de pooling e classificador denso;
- augmentation por crop/escala, reflexão, jitter fotométrico e random erasing;
- métricas de loss, acurácia, precision, recall, F1, matriz de confusão e tempo;
- melhor checkpoint definido exclusivamente pela loss de validação.


## 1. Protocolo experimental

A divisão `train/val/test` fornecida pelo professor é preservada. O treino contém 300 imagens, com 150 gatos e 150 cães; validação e teste contêm 100 imagens cada, equilibradas entre as classes. Nenhum split aleatório é criado pelo notebook.

O modelo principal desta etapa é a baseline scratch `cnn_baseline_128_sigmoid1`. A comparação opcional de heads inspirados nas MLPs mais fortes da Etapa 1 está documentada em `docs/phase1_cnn_comparison.md`; ela é uma ablação do classificador denso, não uma afirmação de equivalência entre MLP e CNN.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Image, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from cnn_cats_dogs.config import TrainingConfig, get_classifier_preset
from cnn_cats_dogs.data import RGB_MEAN, RGB_STD, build_data_bundle
from cnn_cats_dogs.engine import run_training
from cnn_cats_dogs.model import ScratchCNN, count_trainable_parameters

print(f'Raiz do projeto: {ROOT}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 2. Configuração reprodutível

O notebook não dispara um novo treino por padrão. Primeiro ele lê os artefatos já produzidos em `runs/cnn_scratch_v2/`. Para reproduzir o experimento, altere `RUN_TRAINING` para `True`; uma nova execução será criada em `runs/cnn_scratch_reproduction/`, preservando os resultados finais existentes.


In [ ]:
DATA_DIR = ROOT / 'dataset'
POSITIVE_CLASS = 'dogs'
ARCHITECTURE = 'cnn_baseline_128_sigmoid1'
RUN_TRAINING = False
REPRODUCTION_DIR = ROOT / 'runs' / 'cnn_scratch_reproduction'

RESULT_CANDIDATES = [
    ROOT / 'runs' / 'cnn_scratch_v2',
    ROOT / 'runs' / 'cnn_scratch_balanced',
    REPRODUCTION_DIR,
]
RESULTS_DIR = next(
    (path for path in RESULT_CANDIDATES if (path / 'artifacts' / 'run_summary.json').is_file()),
    REPRODUCTION_DIR,
)

config = TrainingConfig(
    data_dir=DATA_DIR,
    output_dir=REPRODUCTION_DIR,
    positive_class=POSITIVE_CLASS,
    architecture=ARCHITECTURE,
    classifier_dropout=0.10,
    image_size=224,
    batch_size=32,
    epochs=40,
    learning_rate=1e-3,
    weight_decay=1e-4,
    num_workers=8,
    seed=42,
    device='auto',
    use_amp=True,
    early_stopping_patience=8,
)
config.validate()
print('Diretório de resultados selecionado:', RESULTS_DIR)
config


## 3. Dataset e augmentation

O loader valida que os três splits possuem exatamente duas classes e mapeia explicitamente a pasta positiva para o rótulo `1`. O loader de treino embaralha apenas a ordem de amostragem e aplica augmentation. Os loaders de treino limpo, validação e teste usam preprocessamento determinístico.


In [ ]:
bundle = build_data_bundle(config)
print('Classes [0, 1]:', bundle.class_names)
print('Tamanhos:', bundle.sizes)
print('Contagens:', bundle.class_counts)

images, labels = next(iter(bundle.train_loader))

def denormalize_scratch(image: torch.Tensor) -> torch.Tensor:
    mean = torch.tensor(RGB_MEAN).view(3, 1, 1)
    std = torch.tensor(RGB_STD).view(3, 1, 1)
    return (image.cpu() * std + mean).clamp(0, 1)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for axis, image, label in zip(axes.flat, images[:8], labels[:8]):
    axis.imshow(denormalize_scratch(image).permute(1, 2, 0))
    axis.set_title(bundle.class_names[int(label)])
    axis.axis('off')
fig.suptitle('Amostras de treino com augmentation')
fig.tight_layout()


## 4. Arquitetura manual

A representação visual é extraída por três blocos de convoluções `3×3`, BatchNorm, ReLU e MaxPool. Depois de `AdaptiveAvgPool2d(1)`, a rede reduz o mapa espacial a um vetor de 128 features e aplica o classificador denso selecionado.

A baseline usa uma saída binária de um logit e `BCEWithLogitsLoss`. A sigmoide é aplicada apenas para gerar probabilidades nas métricas, preservando estabilidade numérica na loss.


In [ ]:
preset = get_classifier_preset(config.architecture)
model_preview = ScratchCNN(
    hidden_layers=preset.hidden_layers,
    output_mode=preset.output_mode,
    classifier_dropout=config.classifier_dropout,
)
print(preset.description)
print(model_preview)
print(f'Parâmetros treináveis: {count_trainable_parameters(model_preview):,}')


## 5. Treinamento

O motor salva configuração, ambiente, histórico, checkpoint escolhido por validação, gráficos e previsões do teste. Ative o bloco abaixo somente para uma reprodução controlada.


In [ ]:
if RUN_TRAINING:
    artifacts = run_training(config)
    RESULTS_DIR = artifacts.output_dir
    print('Treino concluído:', artifacts.summary_path)
else:
    print('Treino desativado. Lendo artefatos existentes em:', RESULTS_DIR)


## 6. Resultados e artefatos

A célula seguinte lê o resumo da execução. Caso os artefatos ainda não tenham sido copiados para `runs/`, execute a reprodução ou ajuste `RESULTS_DIR`.


In [ ]:
summary_path = RESULTS_DIR / 'artifacts' / 'run_summary.json'
if not summary_path.is_file():
    print(f'Resumo não encontrado: {summary_path}')
else:
    with summary_path.open(encoding='utf-8') as file:
        summary = json.load(file)

    print('Arquitetura:', summary.get('architecture'))
    print('Melhor época:', summary.get('best_epoch'))
    print('Tempo total (s):', round(float(summary.get('training_time_seconds', 0)), 2))
    display(pd.Series(summary['test_metrics'], name='CNN do zero').to_frame())


In [ ]:
learning_curves = RESULTS_DIR / 'plots' / 'learning_curves.png'
confusion_matrix = RESULTS_DIR / 'plots' / 'confusion_matrix_test.png'
history_path = RESULTS_DIR / 'artifacts' / 'history.csv'

for artifact in (learning_curves, confusion_matrix):
    if artifact.is_file():
        display(Image(filename=str(artifact)))

if history_path.is_file():
    history = pd.read_csv(history_path)
    display(history.tail())


## 7. Discussão para o relatório

A CNN treinada do zero atingiu 66,0% de acurácia e F1-score de 68,5% no teste final. A diferença entre treino limpo e validação sugere limitação de generalização compatível com o tamanho reduzido do conjunto de treino.

Esse resultado é o baseline necessário para a Etapa 3: ele mostra que filtros espaciais aprendidos do zero em 300 imagens são insuficientes para competir com representações pré-treinadas em ImageNet. A comparação completa e os cuidados metodológicos finais estão em `docs/final_results.md`.
